In [2034]:
import numpy as np
import pandas as pd
import plotly.express as px
import collections
import itertools
import tqdm
from copy import deepcopy

In [2035]:
np.random.seed(0)

In [2036]:
def bayesian_update(priors):
    if np.sum(priors) == 0:
        return np.zeros_like(priors)
    return priors / np.sum(priors)

In [ ]:
def manipulation_thresholds(thresholds, priors, c):
    if np.sum(priors) == 0:
        return thresholds.copy()
    manip_thresholds = np.maximum(0, thresholds - (bayesian_update(priors) / c))
    return manip_thresholds

In [2039]:
def merge_classifiers(thresholds, priors, manip_thresholds):
    if len(thresholds) <= 1:
        return thresholds, priors, manip_thresholds
    
    merged_thresholds = [thresholds[-1]]
    merged_priors = [priors[-1]]
    merged_manip_thresholds = [manip_thresholds[-1]]

    for i in range(len(thresholds)-2,-1,-1):
        if manip_thresholds[i] < merged_manip_thresholds[-1]:
            merged_thresholds.append(thresholds[i])
            merged_priors.append(priors[i])
            merged_manip_thresholds.append(manip_thresholds[i])
        else:
            merged_priors[-1] += priors[i]
    
    return np.array(merged_thresholds)[::-1], np.array(merged_priors)[::-1], np.array(merged_manip_thresholds)[::-1]

In [2040]:
def balance_priors(priors, random=True):
    total = np.sum(priors)
    if total == 1:
        return priors
    indices = priors == 0
    remainder = 1 - total
    if random:
        p = np.random.rand(indices.sum())
        p = (p / p.sum()) * remainder
    else:
        p = remainder / indices.sum()
    priors[indices] = p
    return priors

In [ ]:
def accuracy_loss(thresholds, priors, manip_thresholds, threshold_true):
    losses = []
    for i in range(len(thresholds)):
        loss = np.abs(manip_thresholds[i] - threshold_true)
        losses.append(loss)
    losses = np.array(losses)
    posteriors = bayesian_update(priors)
    return np.dot(losses, posteriors)

In [2042]:
def evaluate_partition(partition, thresholds, priors, threshold_true, c, return_all=False):
    thresholds_p = thresholds[partition]
    priors_p = priors[partition]
    manip_thresholds_p = manipulation_thresholds(thresholds_p, priors_p, c)
    thresholds_p, priors_p, manip_thresholds_p = merge_classifiers(thresholds_p, priors_p, manip_thresholds_p)
    acc_loss_p = accuracy_loss(thresholds_p, priors_p, manip_thresholds_p, threshold_true)
    if return_all:
        return acc_loss_p, thresholds_p, priors_p, manip_thresholds_p
    return acc_loss_p

def evaluate_system(partitions, thresholds, priors, threshold_true, c):
    acc_loss = 0.
    for partition in partitions:
        acc_loss_p = evaluate_partition(partition, thresholds, priors, threshold_true, c)
        acc_loss += acc_loss_p * np.sum(priors[partition])
    return acc_loss

In [2043]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return

    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        yield [[first]] + smaller

In [2044]:
def display_queue(Q, P):
    res = "[  "
    for (a_id, b_id) in Q:
        res += f"({P[a_id]}, {P[b_id]})  "
    res += "]"
    print(res)


def find_partitions_greedy(thresholds, priors, threshold_true, c):
    P = {}
    next_id = 0
    partitions = [[i] for i in range(len(priors))]
    for block in partitions:
        P[next_id] = list(block)
        next_id += 1

    Q = collections.deque(itertools.combinations(P.keys(), 2))
    while Q:
        # display_queue(Q, P)
        a_id, b_id = Q.popleft()
        if a_id not in P.keys() or b_id not in P.keys():
            continue

        a = P[a_id]
        b = P[b_id]

        acc_loss_a = evaluate_partition(a, thresholds, priors, threshold_true, c)
        acc_loss_b = evaluate_partition(b, thresholds, priors, threshold_true, c)
        lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])

        ab = sorted(a + b)
        acc_loss_ab = evaluate_partition(ab, thresholds, priors, threshold_true, c)
        rhs = acc_loss_ab * np.sum(priors[ab])

        if lhs > rhs:
            del P[a_id]
            del P[b_id]

            Q = collections.deque(
                (x, y)
                for (x, y) in Q
                if x not in {a_id, b_id} and y not in {a_id, b_id}
            )

            new_id = next_id
            next_id += 1
            P[new_id] = ab

            for p_id in P.keys():
                if p_id != new_id:
                    Q.append((new_id, p_id))
    return list(P.values())

In [2045]:
def find_partitions_optimal(thresholds, priors, threshold_true, c):
    indices = [i for i in range(len(thresholds))]

    parts = set_partitions(indices)
    partitions_set = []
    for part in parts:
        partitions_set.append(part)

    best_partition = None
    best_loss = np.inf

    for partitions in partitions_set:
        acc_loss = 0.
        for partition in partitions:
            acc_loss_p = evaluate_partition(partition, thresholds, priors, threshold_true, c)
            acc_loss += acc_loss_p * np.sum(priors[partition])

        if acc_loss < best_loss:
            best_loss = acc_loss
            best_partition = deepcopy(partitions)

    return best_partition

In [2046]:
c = 5.0
threshold_true = 0.5

threshold_min, threshold_max, threshold_delta = 0., 1., 0.1
thresholds = np.arange(threshold_min+threshold_delta, threshold_max, threshold_delta).round(4)

# priors = np.zeros_like(thresholds)
# balance_priors(priors, random=True)
priors = np.array([0.11393634, 0.11459784, 0.03594993, 0.25, 0.02203078, 0.05390004, 0.06215049, 0.09743458, 0.25])

print(np.sum(priors))
pd.DataFrame({"threshold": thresholds, "priors": priors}).round(3).T

1.0


,0,1,2,3,4,5,6,7,8
threshold,0.100,0.200,0.300,0.40,0.500,0.600,0.700,0.800,0.90
priors,0.114,0.115,0.036,0.25,0.022,0.054,0.062,0.097,0.25


In [2047]:
partition_greedy = find_partitions_greedy(thresholds, priors, threshold_true, c)
partition_optimal = find_partitions_optimal(thresholds, priors, threshold_true, c)

acc_loss_greedy = evaluate_system(partition_greedy, thresholds, priors, threshold_true, c)
acc_loss_optimal = evaluate_system(partition_optimal, thresholds, priors, threshold_true, c)

In [2048]:
print("Greedy")
print("------")
print(f"Partition: {sorted(partition_greedy)}")
print(f"Acc Loss : {acc_loss_greedy:.4f}")
print()
print("Optimal")
print("-------")
print(f"Partition: {sorted(partition_optimal)}")
print(f"Acc Loss : {acc_loss_optimal:.4f}")

Greedy
------
Partition: [[0, 1, 2, 3, 4, 5], [6], [7], [8]]
Acc Loss : 0.2069

Optimal
-------
Partition: [[0, 1, 2, 3, 4, 5], [6], [7], [8]]
Acc Loss : 0.2069


In [2049]:
def find_mismatches(n, tol=0.):
    for _ in tqdm.trange(1000):
        thresholds = np.sort(np.random.rand(n))
        priors = np.zeros_like(thresholds)
        balance_priors(priors, random=True)
        if abs(1-np.sum(priors)) > 1e-3:
            print(np.sum(priors))
            continue
        
        threshold_true = np.random.rand()
        c = np.random.uniform(0.05, 0.5)

        partition_opt = find_partitions_optimal(thresholds, priors, threshold_true, c)
        partition_greedy = find_partitions_greedy(thresholds, priors, threshold_true, c)

        acc_loss_opt = evaluate_system(partition_opt, thresholds, priors, threshold_true, c)
        acc_loss_greedy = evaluate_system(partition_greedy, thresholds, priors, threshold_true, c)

        if (1 - acc_loss_opt)/(1- acc_loss_greedy) > tol:
        # if acc_loss_greedy - acc_loss_opt > tol:
            return thresholds, priors, threshold_true, c

In [2050]:
n = 10
thresholds_ce, priors_ce, threshold_true_ce, c_ce = find_mismatches(n, tol=1.009)

  0%|          | 2/1000 [00:21<3:02:24, 10.97s/it]


In [2061]:
# c_ce = 0.1214
c_ce = 5.

partition_optimal_ce = find_partitions_optimal(thresholds_ce, priors_ce, threshold_true_ce, c_ce)
partition_greedy_ce = find_partitions_greedy(thresholds_ce, priors_ce, threshold_true_ce, c_ce)

acc_loss_greedy_ce = evaluate_system(partition_greedy_ce, thresholds_ce, priors_ce, threshold_true_ce, c_ce)
acc_loss_optimal_ce = evaluate_system(partition_optimal_ce, thresholds_ce, priors_ce, threshold_true_ce, c_ce)

In [2107]:
c_ce

5.0

In [2062]:
print(f"t*        : {threshold_true_ce:.4f}")
print(f"c         : {c_ce:.4f}")
display(pd.DataFrame({"Thresholds": thresholds_ce, "Priors": priors_ce}).round(4).T)

print("Greedy")
print("------")
print(f"Partition: {sorted(partition_greedy_ce)}")
print(f"Acc Loss : {acc_loss_greedy_ce:.4f}")
print()
print("Optimal")
print("-------")
print(f"Partition: {sorted(partition_optimal_ce)}")
print(f"Acc Loss : {acc_loss_optimal_ce:.4f}")
print()
print(f"Ratio: {(1 - acc_loss_optimal_ce)/(1- acc_loss_greedy_ce):.4f}")

t*        : 0.1966
c         : 5.0000


,0,1,2,3,4,5,6,7,8,9
Thresholds,0.1020,0.1289,0.2104,0.3154,0.3637,0.4386,0.5702,0.6668,0.6706,0.9884
Priors,0.0685,0.0529,0.2141,0.0830,0.1528,0.0801,0.0521,0.0362,0.2151,0.0453


Greedy
------
Partition: [[0, 1, 2, 3, 4], [5], [6], [7], [8], [9]]
Acc Loss : 0.1585

Optimal
-------
Partition: [[0, 1, 2, 3], [4], [5], [6], [7], [8], [9]]
Acc Loss : 0.1521

Ratio: 1.0076


In [2108]:
a, b = [0,1], [2]

acc_loss_a, thresholds_a, priors_a, manip_thresholds_a = evaluate_partition(a, thresholds_ce, priors_ce, threshold_true_ce, c_ce, True)
acc_loss_b, thresholds_b, priors_b, manip_thresholds_b = evaluate_partition(b, thresholds_ce, priors_ce, threshold_true_ce, c_ce, True)

lhs = acc_loss_a * np.sum(priors_ce[a]) + acc_loss_b * np.sum(priors_ce[b])
ab = sorted(a + b)

acc_loss_ab, thresholds_ab, priors_ab, manip_thresholds_ab = evaluate_partition(ab, thresholds_ce, priors_ce, threshold_true_ce, c_ce, True)
rhs = acc_loss_ab * np.sum(priors_ce[ab])

print(f"                 a: {a}")
print(f"                 b: {b}")
print(f"   accuracy loss a: {acc_loss_a:.4f}")
print(f"   accuracy loss b: {acc_loss_b:.4f}")
print(f"  accuracy loss ab: {acc_loss_ab:.4f}")
print(f"               LHS: {lhs:.4f}")
print(f"               RHS: {rhs:.4f}")
print(f"            merge?: {lhs>rhs}")
print()
print(f"      thresholds a: {thresholds_a.round(4)}")
print(f"          priors a: {priors_a.round(4)}")
print(f"      thresholds b: {thresholds_b.round(4)}")
print(f"          priors b: {priors_b.round(4)}")
print(f"     thresholds ab: {thresholds_ab.round(4)}")
print(f"         priors ab: {priors_ab.round(4)}")
print(f" manip threshold a: {manip_thresholds_a.round(4)}")
print(f" manip threshold b: {manip_thresholds_b.round(4)}")
print(f"manip threshold ab: {manip_thresholds_ab.round(4)}")

                 a: [0, 1]
                 b: [2]
   accuracy loss a: 0.1784
   accuracy loss b: 0.1862
  accuracy loss ab: 0.1182
               LHS: 0.0615
               RHS: 0.0397
            merge?: True

      thresholds a: [0.102  0.1289]
          priors a: [0.0685 0.0529]
      thresholds b: [0.2104]
          priors b: [0.2141]
     thresholds ab: [0.102  0.2104]
         priors ab: [0.0685 0.2669]
 manip threshold a: [0.     0.0418]
 manip threshold b: [0.0104]
manip threshold ab: [0.0612 0.0827]


In [2099]:
a, b = [0,1,2,3], [4]

acc_loss_a, thresholds_a, priors_a, manip_thresholds_a = evaluate_partition(a, thresholds_ce, priors_ce, threshold_true_ce, c_ce, True)
acc_loss_b, thresholds_b, priors_b, manip_thresholds_b = evaluate_partition(b, thresholds_ce, priors_ce, threshold_true_ce, c_ce, True)

lhs = acc_loss_a * np.sum(priors_ce[a]) + acc_loss_b * np.sum(priors_ce[b])
ab = sorted(a + b)

acc_loss_ab, thresholds_ab, priors_ab, manip_thresholds_ab = evaluate_partition(ab, thresholds_ce, priors_ce, threshold_true_ce, c_ce, True)
rhs = acc_loss_ab * np.sum(priors_ce[ab])

print(f"                 a: {a}")
print(f"                 b: {b}")
print(f"   accuracy loss a: {acc_loss_a:.4f}")
print(f"   accuracy loss b: {acc_loss_b:.4f}")
print(f"  accuracy loss ab: {acc_loss_ab:.4f}")
print(f"               LHS: {lhs:.4f}")
print(f"               RHS: {rhs:.4f}")
print(f"            merge?: {lhs>rhs}")
print()
print(f"      thresholds a: {thresholds_a.round(4)}")
print(f"          priors a: {priors_a.round(4)}")
print(f"      thresholds b: {thresholds_b.round(4)}")
print(f"          priors b: {priors_b.round(4)}")
print(f"     thresholds ab: {thresholds_ab.round(4)}")
print(f"         priors ab: {priors_ab.round(4)}")
print(f" manip threshold a: {manip_thresholds_a.round(4)}")
print(f" manip threshold b: {manip_thresholds_b.round(4)}")
print(f"manip threshold ab: {manip_thresholds_ab.round(4)}")

                 a: [0, 1, 2, 3]
                 b: [4]
   accuracy loss a: 0.0936
   accuracy loss b: 0.0329
  accuracy loss ab: 0.0885
               LHS: 0.0442
               RHS: 0.0506
            merge?: False

      thresholds a: [0.102  0.1289 0.2104 0.3154]
          priors a: [0.0685 0.0529 0.2141 0.083 ]
      thresholds b: [0.3637]
          priors b: [0.1528]
     thresholds ab: [0.102  0.1289 0.2104 0.3154 0.3637]
         priors ab: [0.0685 0.0529 0.2141 0.083  0.1528]
 manip threshold a: [0.0693 0.1037 0.1081 0.2757]
 manip threshold b: [0.1637]
manip threshold ab: [0.0781 0.1104 0.1354 0.2864 0.3102]


In [1897]:
def stress_test(n_runs=100):
    incorrect_settings = []
    for _ in tqdm.trange(n_runs):
        n = np.random.randint(2, 11)
        thresholds = np.sort(np.random.rand(n))
        priors = np.zeros_like(thresholds)
        balance_priors(priors, random=True)
        if abs(1-np.sum(priors)) > 1e-3:
            print(np.sum(priors))
            continue
        
        threshold_true = np.random.rand()
        c = np.random.uniform(0.05, 0.5)

        partition_opt = find_partitions_optimal(thresholds, priors, threshold_true, c)
        partition_greedy = find_partitions_greedy(thresholds, priors, threshold_true, c)

        acc_loss_opt = evaluate_system(partition_opt, thresholds, priors, threshold_true, c)
        acc_loss_greedy = evaluate_system(partition_greedy, thresholds, priors, threshold_true, c)

        if (1 - acc_loss_opt)/(1- acc_loss_greedy) < 1:
        # if acc_loss_greedy - acc_loss_opt > tol:
            incorrect_settings.append((thresholds, priors, threshold_true, c))
    return incorrect_settings

In [1900]:
incorrect_settings = stress_test(1000)

100%|██████████| 1000/1000 [16:41<00:00,  1.00s/it]


In [2101]:
len(incorrect_settings)

1

In [2102]:
for incorrect_setting in incorrect_settings:
    thresholds_ic, priors_ic, threshold_true_ic, c_ic = incorrect_setting
    partition_optimal_ic = find_partitions_optimal(thresholds_ic, priors_ic, threshold_true_ic, c_ic)
    partition_greedy_ic = find_partitions_greedy(thresholds_ic, priors_ic, threshold_true_ic, c_ic)

    acc_loss_greedy_ic = evaluate_system(partition_greedy_ic, thresholds_ic, priors_ic, threshold_true_ic, c_ic)
    acc_loss_optimal_ic = evaluate_system(partition_optimal_ic, thresholds_ic, priors_ic, threshold_true_ic, c_ic)

    print(f"t*        : {threshold_true_ic:.4f}")
    print(f"c         : {c_ic:.4f}")
    display(pd.DataFrame({"Thresholds": thresholds_ic, "Priors": priors_ic}).round(4).T)

    print("Greedy")
    print("------")
    print(f"Partition: {sorted(partition_greedy_ic)}")
    print(f"Acc Loss : {acc_loss_greedy_ic}")
    print()
    print("Optimal")
    print("-------")
    print(f"Partition: {sorted(partition_optimal_ic)}")
    print(f"Acc Loss : {acc_loss_optimal_ic}")
    print()
    print(f"Ratio: {(1 - acc_loss_optimal_ic)/(1- acc_loss_greedy_ic):.4f}")

t*        : 0.5248
c         : 0.2270


,0,1,2,3,4
Thresholds,0.1044,0.3851,0.4540,0.5924,0.6937
Priors,0.4510,0.0225,0.0553,0.1602,0.3111


Greedy
------
Partition: [[0, 1], [2, 3], [4]]
Acc Loss : 0.5208305249415951

Optimal
-------
Partition: [[0, 1], [2, 3, 4]]
Acc Loss : 0.5208305249415952

Ratio: 1.0000


In [1846]:
import time

def benchmark_runtime(n):
    times_opt = []
    times_greedy = []
    for _ in tqdm.trange(10):
        thresholds = np.sort(np.random.rand(n))
        priors = np.zeros_like(thresholds)
        balance_priors(priors, random=True)
        if abs(1-np.sum(priors)) > 1e-3:
            print(np.sum(priors))
            break
        
        threshold_true = np.random.rand()
        c = np.random.uniform(0.05, 0.5)

        start_opt = time.perf_counter()
        find_partitions_optimal(thresholds, priors, threshold_true, c)
        end_opt = time.perf_counter()
        
        start_greedy = time.perf_counter()
        find_partitions_greedy(thresholds, priors, threshold_true, c)
        end_greedy = time.perf_counter()

        times_opt.append(end_opt - start_opt)
        times_greedy.append(end_greedy - start_greedy)

    return np.mean(times_opt), np.mean(times_greedy)

In [1847]:
times_opt = []
times_greedy = []
for i in range(1,11):
    time_opt, time_greedy = benchmark_runtime(i)
    times_opt.append(time_opt)
    times_greedy.append(time_greedy)

100%|██████████| 10/10 [01:17<00:00,  7.80s/it]


In [1848]:
results = {"n": [i for i in range(1,11)], "opt": times_opt, "greedy": times_greedy}
px.scatter(results, x="n", y=["opt", "greedy"], width=800, height=600, title="Run Time Performance (s)").update_layout(margin=dict(t=50,b=25,l=25,r=25), title=dict(x=0.5))